In [1]:
from sentence_transformers import SentenceTransformer

from ingest import load_faq_data

In [ ]:
from minsearch import VectorSearch
from tqdm.auto import tqdm

import numpy as np

ImportError: cannot import name 'VectorSearchIndex' from 'minsearch' (/workspaces/llm-zoomcamp-2026-code/.venv/lib/python3.12/site-packages/minsearch/__init__.py)

In [43]:
from sqlitesearch import VectorSearchIndex

In [3]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [6]:
d = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering while the form is open. It is not checked against any registered list. Registration is just to gauge interest before the start date."
dv = model.encode(d)

In [7]:
v1.dot(dv)

np.float32(0.39027888)

In [8]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)

np.float32(-0.011185775)

In [9]:
documents = load_faq_data()

In [10]:
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [11]:
texts = []
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [12]:
batch_size = 50
vectors = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i : i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1216

In [13]:
scores = []
for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [14]:
X = np.array(vectors)
scores = X.dot(v1)

Most similar one

In [15]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(559), np.float32(0.762941))

In [16]:
documents[559]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [17]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1] #revert

In [18]:
top5

array([559, 963,  29, 472, 564])

In [19]:
# Faster way
top5 = np.argsort(-scores)[:5]

In [21]:
vindex = VectorSearch(keyword_fields = ["course"])
vindex.fit(X, documents)

In [24]:
vindex.search(v1, num_results = 5, filter_dict = {"course": "data-engineering-zoomcamp"})

[{'id': '3f1424af17',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."},
 {'id': '068529125b',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'},
 {'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'questi

In [30]:
from ingest import load_faq_data, build_index
from dotenv import load_dotenv
from openai import OpenAI

from rag_helper import RAGBase

In [33]:
documents = load_faq_data()

load_dotenv()
openai_client = OpenAI()

index = build_index(documents)

In [34]:
assistant = RAGBase(
    index = index,
    llm_client = openai_client
)

In [35]:
query = "I just found out about the program, can I still enroll?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.'

In [37]:
class RAGVector(RAGBase):
    
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder
    
    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}
        
        return self.index.search(
            query_vector,
            filter_dict = filter_dict,
            num_results=num_results
        )
        

In [40]:
vector_assistant = RAGVector(
    embedder=model,
    index = vindex,
    llm_client = openai_client
)

In [41]:
vector_assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [44]:
vs_index = VectorSearchIndex(
    keyword_fields = ["course"],
    mode = "ivf",
    db_path = "faq_vectors2.db"
)

In [45]:
vs_index.fit(vectors, documents)

In [46]:
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results = 5)

In [48]:
vs_index.close()

In [47]:
print(results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}, {'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General C